# FEEDS — Manuscript statistical analysis

Produces, for the manuscript:

1. **Superiority** — FEEDS vs any comparator arm you declare (random sampling, DPP, KMeans, pseudolabels, …).
2. **Non-inferiority** — FEEDS vs a reference arm (e.g. 100% labeled).

Comparator arms are fully configurable in the `CONFIG` cell — add as many as you like, each with its own dataset, fold(s), and test kind.

Both are paired at the case level and reported:
- **pooled** across the chosen test sets, AND **stratified within groups** (diagnosis, tracer, anatomic region) exactly like Table 3 in the paper;
- with the 5 random iterations summarised **both ways side by side** (per-case mean and per-case median);
- with **raw + multiplicity-corrected** p-values (Holm and Benjamini-Hochberg FDR) across groups.

A single **dataset switch** (`WHICH`) runs the analysis on AutoPET, DeepPSMA, DH, or all pooled.

All logic is in `scripts/feeds_stats_core.py`.

## 0 · Setup

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
import os, sys
sys.path.insert(0, 'scripts')
import numpy as np, pandas as pd
import feeds_stats_core as C

os.environ['nnUNet_results'] = '//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/'
assert 'nnUNet_results' in os.environ, "set os.environ['nnUNet_results'] first"
METADATA = 'fdg_metadata.csv'
SPLIT    = 'test_predictions'

## 1 · Arm configuration + dataset switch

Fold ↔ %-labelled mapping (from `Dataset_Key.md`): Dataset 111 = FEEDS (`fold_4`=30%, `fold_9`=100%); Dataset 330 = 5× random @30% (`fold_0..fold_4`).

`WHICH` is the dataset switch: `'ALL'`, a single tag, or a list of tags.

In [29]:
CONFIG = [
    dict(tag='AutoPET',
         feeds=(111, 'fold_4'),            # FEEDS reference arm (30%)
         comparators=[
             # each comparator: name + dataset + fold(s) + test kind + how to aggregate
             dict(name='random', dataset=330,
                  folds=['fold_0','fold_1','fold_2','fold_3','fold_4'],
                  kind='superiority',    agg='iterations'),  # 5 iters -> mean+median
             dict(name='full',   dataset=999, folds='fold_9',
                  kind='noninferiority', agg='single'),      # 100% labeled
             # --- add any other arms to test FEEDS against ---
             # dict(name='dpp',    dataset=444, folds='fold_4', kind='superiority', agg='single'),
             # dict(name='kmeans', dataset=700, folds='fold_2', kind='superiority', agg='single'),
             # dict(name='ssl_pl', dataset=222, folds='fold_11', kind='superiority', agg='single'),
         ]),
    # Add DeepPSMA / DH as separate datasets with the same comparator list if needed.
]

# agg: 'single' = one fold as-is; 'iterations' = pool multiple folds per case (mean+median).
# kind: 'superiority' (paired t + Wilcoxon) or 'noninferiority' (needs a margin below).
# Legacy shorthand still works: random=(330,[folds]) and full=(111,'fold_9').

# ---- DATASET SWITCH: 'ALL' | 'AutoPET' | 'DeepPSMA' | 'DH' | ['AutoPET','DH'] ----
WHICH = "ALL"
SPLIT = "DHMC_data_predictions"

## 2 · Pre-specified non-inferiority margins

Fix a priori on clinical grounds (confirm with clinical co-authors). Placeholders below.

In [30]:
MARGINS = {
    'Dice':          0.05,   # Dice points
    'false_pos_vol': 5.0,    # mL
    'false_neg_vol': 5.0,    # mL
}

## 3 · Run everything

`run_all` loops over **every comparator** declared in `CONFIG` (FEEDS is always the reference), and for each random-summary (mean and median) prints:
- the comparator's test (superiority or non-inferiority) pooled + by diagnosis + by tracer.

Returns a nested dict `results[summary][(comparator_name, kind, by)] = dataframe`.

In [31]:
results = C.run_all(
    CONFIG, MARGINS, which=WHICH, split=SPLIT, metadata_csv=METADATA,
    group_bys=('diagnosis', 'tracer'),   # add more if desired
    summaries=('mean', 'median'),        # both, side by side
)


Datasets: ['AutoPET']  |  pooled cases: 23
  AutoPET: 23
Comparators: [('random', 'superiority'), ('full', 'noninferiority')]

RANDOM SUMMARY = MEAN

>>> SUPERIORITY — FEEDS vs random  [pooled]
metric group  n  feeds_mean  comp_val     delta         t  p_ttest  p_wilcoxon favored
  Dice   ALL 23    0.534420  0.491455  0.042965  2.679474 0.013694    0.008262   FEEDS
 FPVol   ALL 23   19.080612 23.282582 -4.201970 -1.123455 0.273355    0.027674   FEEDS
 FNVol   ALL 23    2.720857  2.889170 -0.168313 -1.116507 0.276252    0.789675   FEEDS

>>> SUPERIORITY — FEEDS vs random  [by diagnosis]
Empty DataFrame
Columns: []
Index: []

>>> SUPERIORITY — FEEDS vs random  [by tracer]
Empty DataFrame
Columns: []
Index: []

>>> NON-INFERIORITY — FEEDS vs full  [pooled]
metric group  n  margin  feeds_mean   ref_val      delta       t_ni     p_noninf  noninferior
  Dice   ALL 23    0.05    0.534420  0.488688   0.045732   4.876533 3.553695e-05         True
 FPVol   ALL 23    5.00   19.080612 34.727814 -1

## 4 · Anatomic-region stratification (Table 3-style, by organ)

Region metrics come from the `bone_analysis` `*_mets_metrics.csv` files, loaded per case
per organ (liver / lung / bone / hr_bone). We build FEEDS, random (mean+median), and 100%
region arms, then run the same paired tests stratified `by='organ'`.

Set `C.PRIORITY_BONES` to the notebook's list if you want the `hr_bone` group.

In [ ]:
cfg = C.select_datasets(CONFIG, WHICH)

# region arms need the case roster to filter the per-case CSVs
def region_arm(dataset, folds, tag):
    voxel = C.load_arm(dataset, folds if isinstance(folds, list) else [folds],
                       SPLIT, METADATA, dataset_tag=tag)
    names = voxel['case_name'].unique().tolist()
    return C.load_region_arm(dataset, folds if isinstance(folds, list) else [folds],
                             SPLIT, names, METADATA, dataset_tag=tag)

feeds_reg, rand_reg_long, full_reg = [], [], []
for ds in cfg:
    feeds_reg.append(region_arm(ds['feeds'][0], ds['feeds'][1], ds['tag']))
    rand_reg_long.append(region_arm(ds['random'][0], ds['random'][1], ds['tag']))
    if ds.get('full'):
        full_reg.append(region_arm(ds['full'][0], ds['full'][1], ds['tag']))

feeds_reg = pd.concat(feeds_reg, ignore_index=True)
# pool the 5 random iterations per (case, organ), keeping mean + median
rand_reg  = C.pool_random_by_case(pd.concat(rand_reg_long, ignore_index=True), extra_group='organ')
full_reg  = pd.concat(full_reg, ignore_index=True) if full_reg else pd.DataFrame()

if not feeds_reg.empty:
    print('### REGION superiority (FEEDS vs random), by organ — MEAN ###')
    print(C.superiority_test(feeds_reg, rand_reg, by='organ', summary='mean').to_string(index=False))
    print('\n### REGION superiority, by organ — MEDIAN ###')
    print(C.superiority_test(feeds_reg, rand_reg, by='organ', summary='median').to_string(index=False))
    if not full_reg.empty:
        print('\n### REGION non-inferiority (FEEDS vs 100%), by organ — MEAN ###')
        print(C.noninferiority_test(feeds_reg, full_reg, MARGINS, by='organ', summary='mean').to_string(index=False))
else:
    print('No region data found on disk for the selected datasets.')

## 5 · Save all result tables

In [ ]:
os.makedirs('outputs', exist_ok=True)
for summ, d in results.items():
    for (name, kind, by), df in d.items():
        df.to_csv(f'outputs/{name}_{kind}_{by}_{summ}.csv', index=False)
print('saved all comparator tables (each arm x kind x grouping x summary) to outputs/')